In [ ]:
import spatialdata as sd
import spatialdata_plot
import tifffile
from spatialdata.models import Labels2DModel, Image2DModel
from spatialdata.transformations import Translation, get_transformation, remove_transformation
import dask.array as da

# Dataset for Tutorial notebook 1

spatialdata for spatialdata introduction, visualization, QC and deconvolution
1. images from spaceranger (only high resolution one)
2. 16um-bin table 
3. 16um-bin positions


In [ ]:
# zarr path to the P5 dataset
zarr_path = "/workspaces/data/P5.zarr"

# load the dataset using the read_zarr function
sdata = sd.read_zarr(
    zarr_path
)

sdata

In [ ]:
del sdata.images["P5_lowres_image"]
del sdata.shapes["P5_square_002um"]
del sdata.tables["square_002um"]
del sdata.shapes["P5_square_008um"]
del sdata.tables["square_008um"]

In [ ]:
cs_to_remove = {"P5_downscaled_hires", "P5_downscaled_lowres"}

for element_type, name, element in sdata.gen_spatial_elements():
    transformations = get_transformation(element, get_all=True)

    for cs in cs_to_remove:
        if cs in transformations:
            remove_transformation(element, to_coordinate_system=cs)

In [ ]:
sdata

In [ ]:
out_path = "/workspaces/modified_data/P5_full_slide.zarr"

sdata.write(
    out_path,
    overwrite=True,
)

In [ ]:
del sdata

# Dataset for Tutorial notebook 2

spatialdata for showing segmentation results, close-up on a small portion of the slide
1. microscopy image
2. segmentation mask
3. 2um-bin table 
4. 2um-bin position annotation


In [ ]:
# Mask coordinates in the P5 frame of reference
x_min, x_max = 59000, 63000
y_min, y_max = 55000, 62000

In [ ]:
# zarr path to the P5 dataset
zarr_path = "/workspaces/data/P5.zarr"

# load the dataset using the read_zarr function
sdata = sd.read_zarr(
    zarr_path
)

sdata

In [ ]:
# Extract from the SpatialData the portion corresponding to the mask.

sdata_roi = sdata.query.bounding_box(
    axes=("x", "y"),
    min_coordinate=[x_min, y_min],
    max_coordinate=[x_max, y_max],
    target_coordinate_system="P5",
)

sdata_roi

In [ ]:
mask_path = "/workspaces/original_data/areas_segmentation_mask.tif"
mask_label_name = "aoi_segmentation_mask"

areas_mask = tifffile.imread(mask_path)

expected_shape = (y_max - y_min, x_max - x_min)
if areas_mask.shape != expected_shape:
    raise ValueError(
        f"Mask has shape {areas_mask.shape}, but the P5 region requires {expected_shape}."
    )

sdata_roi.labels[mask_label_name] = Labels2DModel.parse(
    areas_mask,
    dims=("y", "x"),
    transformations={
        "P5": Translation([x_min, y_min], axes=("x", "y")),
    },
)

sdata_roi

In [ ]:
crop_path = "/workspaces/original_data/Visium_HD_Human_Colon_Cancer_P5_tissue_image_crop_x59000-63000_y55000-62000.tif"

store = tifffile.imread(crop_path, aszarr=True, key=0)
img = da.from_zarr(store)

sdata_roi.images["P5_crop_image"] = Image2DModel.parse(
    img,
    dims=("y", "x", "c"),
    c_coords=["r", "g", "b"],
    transformations={
        "P5": Translation([59000, 55000], axes=("x", "y"))
    },
)

sdata_roi

In [ ]:
del sdata_roi.images["P5_hires_image"]
del sdata_roi.images["P5_lowres_image"]
del sdata_roi.shapes["P5_square_016um"]
del sdata_roi.tables["square_016um"]
del sdata_roi.shapes["P5_square_008um"]
del sdata_roi.tables["square_008um"]

In [ ]:
cs_to_remove = {"P5_downscaled_hires", "P5_downscaled_lowres"}

for element_type, name, element in sdata_roi.gen_spatial_elements():
    transformations = get_transformation(element, get_all=True)

    for cs in cs_to_remove:
        if cs in transformations:
            remove_transformation(element, to_coordinate_system=cs)

In [ ]:
sdata_roi

In [ ]:
out_path = "/workspaces/modified_data/P5_crop_slide.zarr"

sdata_roi.write(
    out_path,
    overwrite=True,
)

In [ ]:
del sdata
del sdata_roi

# Plots

In [ ]:
# zarr path to the P5 dataset
zarr_path = "/workspaces/modified_data/P5_full_slide.zarr"

# load the dataset using the read_zarr function
sdata = sd.read_zarr(
    zarr_path
)

sdata

In [ ]:
sdata.pl.render_images(
    'P5_hires_image',
).pl.show(
    coordinate_systems="P5",
    title="CytAssist image",
)

In [ ]:
sdata.pl.render_images(
    'P5_hires_image',
    cmap="gray",
).pl.render_shapes(
    "P5_square_016um",
    color="#00000000",
    fill_alpha=0,
    outline_color="yellow",
    outline_width=0.01,
    outline_alpha=0.4,
).pl.show(
    coordinate_systems="P5",
    title="CytAssist image",
)


In [ ]:
del sdata

In [ ]:
# zarr path to the P5 dataset
zarr_path = "/workspaces/modified_data/P5_crop_slide.zarr"

# load the dataset using the read_zarr function
sdata_roi = sd.read_zarr(
    zarr_path
)

sdata_roi

In [ ]:
sdata_roi.pl.render_images(
    "P5_crop_image",
).pl.show(
    coordinate_systems="P5",
    title="P5 ROI",
)

In [ ]:
sdata_roi.pl.render_images(
    "P5_crop_image",
    cmap="gray",
).pl.render_labels(
    "aoi_segmentation_mask",
    fill_alpha=0,
    outline_alpha=1,
    outline_color="red",
    na_color="red",
).pl.show(
    coordinate_systems="P5",
    title="P5 ROI with segmentation mask",
)

In [ ]:
del sdata_roi